# Small-grid inverter: native DPsim vs FMI Model Exchange and Co-Simulation

This notebook compares three fresh, closed-loop DPsim simulations on the same source–PI-line–PCC inverter grid: a native nonlinear MNA inverter, FMI Model Exchange (ME), and FMI Co-Simulation (CS). **Native DPsim is the reference for every error calculation.** No recorded trace is replayed into a simulation.

Each run lasts 5 s and applies a 50 MW PCC load step at 2.5 s. Retained fixed timesteps are 25, 50, 100, 500, and 1000 µs. All three use the same grid, initialization, inverter parameters, and output sampling.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

cwd = Path.cwd().resolve()
repo = next(
    (
        p
        for p in (cwd, *cwd.parents)
        if (p / "dpsim-fmi/examples/small_inverter_grid").exists()
    ),
    None,
)
if repo is None:
    raise FileNotFoundError("Open this notebook from the DPsim repository.")
data = repo / "build-fmi/small-inverter-grid/results"
base = json.loads((data / "metrics.json").read_text())
sweep = json.loads((data / "metrics_sweep.json").read_text())
native = json.loads((data / "metrics_native_dpsim.json").read_text())
case_keys = {
    25: "main",
    50: "dt_50us",
    100: "dt_100us",
    500: "dt_500us",
    1000: "dt_1000us",
}
records = {}
for us, key in case_keys.items():
    item = base[key] if key == "main" else sweep[key]
    nm = native[key]
    traces = {
        "DPsim native": pd.read_csv(nm["native_trace"]),
        "DPsim ME": pd.read_csv(item["traces"]["DPsim ME"]),
        "DPsim CS": pd.read_csv(item["traces"]["DPsim CS"]),
    }
    ref = traces["DPsim native"]
    ts = ref.time_s.to_numpy()
    dt = us * 1e-6
    if len(ref) != round(5 / dt) + 1:
        raise ValueError(f"wrong sample count at {us} µs")
    for name, f in traces.items():
        if len(f) != len(ref) or not np.allclose(
            f.time_s, ts, rtol=0, atol=max(1e-12, dt * 1e-8)
        ):
            raise ValueError(f"time-grid mismatch for {name}, {us} µs")
        if not np.isfinite(f.select_dtypes(include=[np.number]).to_numpy()).all():
            raise ValueError(f"non-finite data: {name}")
    records[us] = {
        "key": key,
        "case": item["case"],
        "metrics": item,
        "native": nm,
        "traces": traces,
    }
steps = sorted(records)
methods = ["DPsim native", "DPsim ME", "DPsim CS"]
colors = {"DPsim native": "#9467bd", "DPsim ME": "#1769aa", "DPsim CS": "#d55e00"}
display(
    pd.DataFrame(
        {
            "timestep (µs)": steps,
            "sample count": [len(records[x]["traces"]["DPsim native"]) for x in steps],
        }
    )
)

## Full-run RMS errors against native DPsim

Errors use all saved samples and all three phases. Percentages are normalized by the native DPsim RMS for that quantity and timestep. Native DPsim is zero by definition.

In [ ]:
rows = []
for us in steps:
    ref = records[us]["traces"]["DPsim native"]
    for method in methods:
        f = records[us]["traces"][method]
        for quantity, prefix, unit in (
            ("PCC voltage", "v", "V"),
            ("Inverter current", "i", "A"),
        ):
            cols = [f"{prefix}_{p}_{unit}" for p in "abc"]
            d = f[cols].to_numpy() - ref[cols].to_numpy()
            rv = ref[cols].to_numpy()
            rms = float(np.sqrt(np.mean(d * d)))
            denom = float(np.sqrt(np.mean(rv * rv)))
            rows.append(
                {
                    "timestep (µs)": us,
                    "method": method,
                    "quantity": quantity,
                    f"RMS error ({unit})": rms,
                    "RMS error (% native DPsim RMS)": 100 * rms / denom,
                    f"max absolute error ({unit})": float(np.max(np.abs(d))),
                }
            )
errors = pd.DataFrame(rows)
display(
    errors.style.format(
        {
            "RMS error (V)": "{:,.6g}",
            "RMS error (A)": "{:.6g}",
            "RMS error (% native DPsim RMS)": "{:.6g}",
            "max absolute error (V)": "{:,.6g}",
            "max absolute error (A)": "{:.6g}",
        }
    )
)
fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
for m in methods:
    q = errors[errors.method == m]
    for a, quantity in ((ax[0], "PCC voltage"), (ax[1], "Inverter current")):
        z = q[q.quantity == quantity].sort_values("timestep (µs)")
        a.plot(
            z["timestep (µs)"],
            z["RMS error (% native DPsim RMS)"],
            marker="o",
            color=colors[m],
            label=m,
        )
for a in ax:
    a.set_xscale("log")
    a.set_xticks(steps, labels=[str(x) for x in steps])
    a.grid(True, which="both", alpha=0.3)
    a.legend()
ax[0].set_ylabel("Voltage RMS error [% native DPsim RMS]")
ax[1].set_ylabel("Current RMS error [% native DPsim RMS]")
ax[1].set_xlabel("Fixed step [µs]")
fig.suptitle("Three-phase RMS error over each 5 s run")
fig.tight_layout()
plt.show()
plt.close(fig)

## Waveforms and error traces at every timestep

Each timestep has separate ME and CS figures. Rows show phase-a traces, signed pointwise differences, and centered 100 ms rolling RMS error. The common load step is marked in each panel.

In [ ]:
for us in steps:
    rec = records[us]
    tr = rec["traces"]
    ref = tr["DPsim native"]
    t = ref.time_s.to_numpy()
    dt = t[1] - t[0]
    window = max(3, int(round(0.1 / dt)))
    event = float(rec["case"]["load_step_s"])
    stride = max(1, len(t) // 18000)
    rv = float(np.sqrt(np.mean(ref.v_a_V.to_numpy() ** 2)))
    ri = float(np.sqrt(np.mean(ref.i_a_A.to_numpy() ** 2)))
    for method in ("DPsim ME", "DPsim CS"):
        f = tr[method]
        dv = f.v_a_V.to_numpy() - ref.v_a_V.to_numpy()
        di = f.i_a_A.to_numpy() - ref.i_a_A.to_numpy()
        wv = (
            100
            * np.sqrt(
                pd.Series(dv**2)
                .rolling(window, center=True, min_periods=1)
                .mean()
                .to_numpy()
            )
            / rv
        )
        wi = (
            100
            * np.sqrt(
                pd.Series(di**2)
                .rolling(window, center=True, min_periods=1)
                .mean()
                .to_numpy()
            )
            / ri
        )
        fig, ax = plt.subplots(3, 2, figsize=(13, 9), sharex=True)
        ax[0, 0].plot(
            t[::stride],
            ref.v_a_V.to_numpy()[::stride] / 1e3,
            "--",
            color=colors["DPsim native"],
            label="DPsim native",
        )
        ax[0, 0].plot(
            t[::stride],
            f.v_a_V.to_numpy()[::stride] / 1e3,
            color=colors[method],
            label=method,
        )
        ax[0, 1].plot(
            t[::stride],
            ref.i_a_A.to_numpy()[::stride],
            "--",
            color=colors["DPsim native"],
            label="DPsim native",
        )
        ax[0, 1].plot(
            t[::stride],
            f.i_a_A.to_numpy()[::stride],
            color=colors[method],
            label=method,
        )
        ax[1, 0].plot(t[::stride], dv[::stride] / 1e3, color=colors[method])
        ax[1, 1].plot(t[::stride], di[::stride], color=colors[method])
        ax[2, 0].plot(t[::stride], wv[::stride], color=colors[method])
        ax[2, 1].plot(t[::stride], wi[::stride], color=colors[method])
        labels = (
            ("PCC phase-a [kV]", "Inverter phase-a [A]"),
            ("Signed voltage difference [kV]", "Signed current difference [A]"),
            ("100 ms RMS error [% native RMS]", "100 ms RMS error [% native RMS]"),
        )
        for r in range(3):
            for c in range(2):
                ax[r, c].set_ylabel(labels[r][c])
                ax[r, c].axvline(event, color="0.35", ls=":")
                ax[r, c].grid(True, alpha=0.28)
        ax[0, 0].legend()
        ax[0, 1].legend()
        ax[2, 0].set_xlabel("Time [s]")
        ax[2, 1].set_xlabel("Time [s]")
        fig.suptitle(f"{us} µs: {method} vs native DPsim | 5 s")
        fig.tight_layout()
        plt.show()
        plt.close(fig)

## Combined error traces and direct pairwise differences

The first plot overlays ME and CS rolling error against native DPsim. The table and second set of plots compare the methods pairwise without another reference.

In [ ]:
pair_rows = []
pairs = (
    ("Native − ME", "DPsim native", "DPsim ME"),
    ("Native − CS", "DPsim native", "DPsim CS"),
    ("ME − CS", "DPsim ME", "DPsim CS"),
)
pair_colors = {"Native − ME": "#9467bd", "Native − CS": "#008b68", "ME − CS": "#d55e00"}
for us in steps:
    rec = records[us]
    tr = rec["traces"]
    ref = tr["DPsim native"]
    t = ref.time_s.to_numpy()
    dt = t[1] - t[0]
    window = max(3, int(round(0.1 / dt)))
    stride = max(1, len(t) // 14000)
    event = float(rec["case"]["load_step_s"])
    fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
    for method in ("DPsim ME", "DPsim CS"):
        f = tr[method]
        for axis, sig, denom, title in (
            (
                ax[0],
                "v_a_V",
                float(np.sqrt(np.mean(ref.v_a_V.to_numpy() ** 2))),
                "PCC phase-a voltage",
            ),
            (
                ax[1],
                "i_a_A",
                float(np.sqrt(np.mean(ref.i_a_A.to_numpy() ** 2))),
                "Inverter phase-a current",
            ),
        ):
            d = f[sig].to_numpy() - ref[sig].to_numpy()
            roll = (
                100
                * np.sqrt(
                    pd.Series(d * d)
                    .rolling(window, center=True, min_periods=1)
                    .mean()
                    .to_numpy()
                )
                / denom
            )
            axis.plot(t[::stride], roll[::stride], color=colors[method], label=method)
            axis.set_ylabel(f"{title} error [% native RMS]")
    for axis in ax:
        axis.axvline(event, color="0.35", ls=":", label="load step")
        axis.grid(True, alpha=0.3)
        axis.legend()
    ax[1].set_xlabel("Time [s]")
    fig.suptitle(f"{us} µs: ME and CS errors vs native DPsim")
    fig.tight_layout()
    plt.show()
    plt.close(fig)
    for label, left, right in pairs:
        a = tr[left]
        b = tr[right]
        for prefix, unit, q in (
            ("v", "V", "PCC voltage"),
            ("i", "A", "Inverter current"),
        ):
            cols = [f"{prefix}_{p}_{unit}" for p in "abc"]
            d = a[cols].to_numpy() - b[cols].to_numpy()
            r = ref[cols].to_numpy()
            pair_rows.append(
                {
                    "timestep (µs)": us,
                    "pair": label,
                    "quantity": q,
                    "RMS difference (% native DPsim RMS)": 100
                    * np.sqrt(np.mean(d * d))
                    / np.sqrt(np.mean(r * r)),
                }
            )
        fig, ax = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
        for axis, sig, denom, title in (
            (
                ax[0],
                "v_a_V",
                float(np.sqrt(np.mean(ref.v_a_V.to_numpy() ** 2))),
                "PCC voltage",
            ),
            (
                ax[1],
                "i_a_A",
                float(np.sqrt(np.mean(ref.i_a_A.to_numpy() ** 2))),
                "Inverter current",
            ),
        ):
            d = a[sig].to_numpy() - b[sig].to_numpy()
            roll = (
                100
                * np.sqrt(
                    pd.Series(d * d)
                    .rolling(window, center=True, min_periods=1)
                    .mean()
                    .to_numpy()
                )
                / denom
            )
            axis.plot(t[::stride], roll[::stride], color=pair_colors[label])
            axis.set_ylabel(f"{title} difference [% native RMS]")
            axis.grid(True, alpha=0.3)
        ax[1].set_xlabel("Time [s]")
        fig.suptitle(f"{us} µs: {label}")
        fig.tight_layout()
        plt.show()
        plt.close(fig)
display(
    pd.DataFrame(pair_rows)
    .pivot(
        index=["timestep (µs)", "pair"],
        columns="quantity",
        values="RMS difference (% native DPsim RMS)",
    )
    .style.format("{:.6g}")
)

## Runtime and solver work

Whole-run time is shown over the timestep sweep. The stage chart reports component pre-step, solver, post-step, and logging task time for the 25 µs run. Native and ME Newton iteration counts are listed separately; CS performs one communication step per timestep.

In [ ]:
timing_rows = []
for us in steps:
    rec = records[us]
    key = rec["key"]
    for method, mode in (
        ("DPsim native", "native"),
        ("DPsim ME", "me"),
        ("DPsim CS", "cs"),
    ):
        timing = (
            rec["native"]["timing"]
            if mode == "native"
            else json.loads(
                (data / "dpsim" / mode / key / f"dpsim_{mode}_timing.json").read_text()
            )
        )
        timing_rows.append(
            {
                "timestep (µs)": us,
                "method": method,
                "wall time (s)": timing["process_wall_time_s"],
            }
        )
timing_sweep = pd.DataFrame(timing_rows)
display(
    timing_sweep.pivot(
        index="timestep (µs)", columns="method", values="wall time (s)"
    ).style.format("{:.3f}")
)
fig, ax = plt.subplots(figsize=(10, 5))
for method in methods:
    z = timing_sweep[timing_sweep.method == method].sort_values("timestep (µs)")
    ax.plot(
        z["timestep (µs)"],
        z["wall time (s)"],
        marker="o",
        color=colors[method],
        label=method,
    )
ax.set_xscale("log")
ax.set_xticks(steps, labels=[str(x) for x in steps])
ax.set_xlabel("Fixed step [µs]")
ax.set_ylabel("Whole-run wall time [s]")
ax.set_title("5 s runtime by timestep")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()
plt.close(fig)

stage_rows = []
for mode, method, key in (
    ("native", "DPsim native", "dt_25us"),
    ("me", "DPsim ME", "main"),
    ("cs", "DPsim CS", "main"),
):
    tasks = pd.read_csv(
        data
        / "dpsim"
        / mode
        / key
        / "logs"
        / f"SmallGrid_Inverter_{mode}"
        / f"SmallGrid_Inverter_{mode}_TaskTimes.csv"
    )
    for label, suffix in (
        ("Component pre-step", ".MnaPreStep"),
        ("Solver task", ".Solve"),
        ("Component post-step", ".MnaPostStep"),
        ("Write/logging", ".Write"),
    ):
        stage_rows.append(
            {
                "method": method,
                "stage": label,
                "seconds": float(
                    tasks[tasks.task.str.endswith(suffix)].total_ns.sum() / 1e9
                ),
            }
        )
    timing = (
        records[25]["native"]["timing"]
        if mode == "native"
        else json.loads(
            (data / "dpsim" / mode / "main" / f"dpsim_{mode}_timing.json").read_text()
        )
    )
    stage_rows.append(
        {"method": method, "stage": "Whole-run wall", "seconds": timing["wall_s"]}
    )
stages = pd.DataFrame(stage_rows)
display(
    stages.pivot(index="method", columns="stage", values="seconds")
    .fillna(0)
    .style.format("{:.4f}")
)
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
stages[stages.stage != "Whole-run wall"].pivot(
    index="method", columns="stage", values="seconds"
).fillna(0).plot(kind="bar", stacked=True, ax=ax[0])
wall = stages[stages.stage == "Whole-run wall"]
ax[1].bar(wall.method, wall.seconds, color=[colors[x] for x in wall.method])
ax[0].set_title("DPsim component and solver stages")
ax[0].set_ylabel("Accumulated task time [s]")
ax[1].set_title("25 µs run")
ax[1].set_ylabel("Whole-run wall time [s]")
for a in ax:
    a.tick_params(axis="x", rotation=20)
    a.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()
plt.close(fig)

iteration_rows = []
for us in steps:
    n = records[us]["native"]["newton"]
    iteration_rows.append(
        {
            "method": "DPsim native",
            "timestep (µs)": us,
            "steps": n["accepted_steps"],
            "min": n["min_iterations"],
            "median": n["median_iterations"],
            "p95": n["p95_iterations"],
            "max": n["max_iterations"],
            "histogram": n["iteration_histogram"],
            "max scaled residual": n["max_scaled_residual"],
        }
    )
    q = records[us]["metrics"].get("newton_convergence")
    if q:
        iteration_rows.append(
            {
                "method": "DPsim ME",
                "timestep (µs)": us,
                "steps": q["steps"],
                "min": q["min_iterations"],
                "median": q["median_iterations"],
                "p95": q["p95_iterations"],
                "max": q["max_iterations"],
                "histogram": q["iteration_histogram"],
                "max scaled residual": q["max_scaled_state_residual"],
            }
        )
display(pd.DataFrame(iteration_rows).set_index(["method", "timestep (µs)"]))

## Interpretation

Native DPsim and FMI ME represent the same averaged inverter law and both participate in the DPsim nonlinear network solve. Their direct trace difference measures implementation agreement. FMI CS exchanges the inverter terminal values once per fixed communication step, so its differences from native/ME also include the effect of that explicit coupling.